# Cluster analysis with K-means

Algorisme de l'article [Spectral-clustering approach to Lagrangian vortex detection](https://arxiv.org/pdf/1506.02258) a partir de trajectòries del sistema dinàmic donat 
pel següent sistema d'EDOs:  $x'=y + \epsilon f(t)$,  $y'=x-x^3$, on $f(t)=sin(t)$.

In [ ]:
import sys
sys.path.append("..")
from src import *
import numpy as np
np.set_printoptions(precision=3, suppress=True)

%load_ext autoreload
%autoreload 2

##### Paràmetres

In [ ]:
dimensio = 2
periode = 4/3 # Periode del sistema de Duffing no autonom
t_span = (0, 30*periode)
t_steps = 500
t_valors = np.linspace(t_span[0], t_span[1], t_steps)
x_min, x_max = (-1.4, 1.4)
y_min, y_max = (-0.9, 0.9)
espai_entre_punts = 0.2
constant_diagonal = 100000
max_clusters = 50

##### 1. Generar $n$ posicions inicials aleatòries

In [ ]:
condicions_inicials = generar_condicions_inicials(
    espai_entre_punts, (x_min, x_max), (y_min, y_max)
)
num_trajectories = len(condicions_inicials)

##### 2. Generar $n$ trajectòries, una per a cada posició inicial

In [ ]:
trajectories = generar_trajectories(edo_duffing_no_autonom, condicions_inicials, t_span, t_valors)
print("(Num trajectories, t_steps, dimensio) =", trajectories.shape)

In [ ]:
grafica_trajectories(trajectories)

In [ ]:
matriu_pesos = calcula_matriu_pesos(trajectories)

##### Opció A: triar radi d'esparsificació tal que el 95% de la matriu de pesos es torni nul·la

In [ ]:
matriu_similaritat_W, sparsification_tol, sparsification_percent = \
    sparcify(matriu_pesos, percent=90)
print(f"S'ha obtingut una esparsificació del "
      f"{sparsification_percent*100:.0f}% usant una tolerància de "
      f"{sparsification_tol:.3f}")
np.fill_diagonal(matriu_similaritat_W, constant_diagonal)
# print(matriu_similaritat_W)

Descomposició espectral: $Lu =\lambda Du$, on la matriu diagonal de graus $D$ és $D_{ii}=\sum _{j=0}^{n} w_{ij}$.

In [ ]:
vaps, veps = calcula_vaps(matriu_similaritat_W, max_clusters)
print("vaps =", vaps)
print("veps.shape =", veps.shape)

In [ ]:
num_clusters, diff_max = calcula_num_clusters_i_max_eigengap(vaps)

In [ ]:
labels = troba_clusters(num_clusters, veps)

In [ ]:
grafica_clusters(condicions_inicials, labels, num_clusters, sparsification_tol, 
                 sparsification_percent, t_steps, t_span, output_dir="../output/no_autonom/")

##### Opció B: triar radi d'esparsificació que maximitza la diferència màxima entre VAPs consecutius

In [ ]:
plot_args = calcula_diffs_vs_radis(matriu_pesos, constant_diagonal, 
                                   condicions_inicials, t_steps, t_span, 
                                   max_clusters=50, num_radis=50)
diffs, nums_clusters, radis, estadistics, sparsificacions, veps_llista = plot_args

In [ ]:
indexs_max_rel = troba_indexs_max_rel(diffs)
print("indexs_max_rel =", indexs_max_rel)

In [ ]:
# indexs_significatius = [5, 10, 20, 35]
# grafica_clusters_maxs_rel(indexs_significatius, radis, sparsificacions, 
#                           nums_clusters, diffs, veps_llista, condicions_inicials, t_steps, t_span)

In [ ]:
grafica_eigengaps_vs_radi(diffs, nums_clusters, radis, estadistics, 
                          sparsificacions, output_dir="../output/no_autonom/")